In [1]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from transformers import pipeline
from ultralytics import FastSAM
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
ry.clear_params()

ry.params_add({'Render/renderShadow': False})

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

generator = pipeline("mask-generation", model="facebook/sam2.1-hiera-large", device=0, return_tensors=True)

Loading weights: 100%|██████████| 741/741 [00:00<00:00, 8609.93it/s]


In [48]:
for i in range(0, 5):
    img_path = f"../samples_of_each_environment/high_clutter/image_dim_{i}.png"
    res = generator(img_path)
    torch.save(res["masks"], f"./SAM/sam_{i}_mask.pt")

In [52]:
# Define an inference source
source = img_path

# Create a FastSAM model
model = FastSAM("FastSAM.pt")  # or FastSAM-x.pt

# Run inference on an image
everything_results = model(source, device="cpu", retina_masks=True, imgsz=1024, conf=0.4, iou=0.9)

# Run inference with bboxes prompt
results = model(source, bboxes=[439, 437, 524, 709])

# Run inference with points prompt
results = model(source, points=[[200, 200]], labels=[1])

# Run inference with texts prompt
results = model(source, texts="a photo of a dog")

# Run inference with bboxes and points and texts prompt at the same time
results = model(source, bboxes=[439, 437, 524, 709], points=[[200, 200]], labels=[1], texts="a photo of a dog")


image 1/1 /media/g3/mahyar/Voxel_manipulation/segmentation/../samples_of_each_environment/high_clutter/image_dim_4.png: 1024x1024 11 objects, 1265.0ms
Speed: 3.2ms preprocess, 1265.0ms inference, 4.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /media/g3/mahyar/Voxel_manipulation/segmentation/../samples_of_each_environment/high_clutter/image_dim_4.png: 1024x1024 1 object, 1256.9ms
Speed: 2.9ms preprocess, 1256.9ms inference, 6.9ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /media/g3/mahyar/Voxel_manipulation/segmentation/../samples_of_each_environment/high_clutter/image_dim_4.png: 1024x1024 (no detections), 1259.1ms
Speed: 2.8ms preprocess, 1259.1ms inference, 4.7ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /media/g3/mahyar/Voxel_manipulation/segmentation/../samples_of_each_environment/high_clutter/image_dim_4.png: 1024x1024 1 object, 1261.5ms
Speed: 2.9ms preprocess, 1261.5ms inference, 1951.0ms postprocess per image at shape (1,

In [53]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'object'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        ...,
 
        [[129, 117, 104],
         [129, 117, 104],
         [129, 117, 104],
         ...,
         [129, 117, 104],
         [129, 117, 104],
         [129, 117, 104]],
 
        [[129, 117, 104],
         [129, 117, 104],
         [129, 117, 104

In [5]:
import numpy as np
from sklearn.cluster import KMeans
from PIL import Image


def color_segmentation(image, n_segments=5):
    if isinstance(image, str):
        image = np.array(Image.open(image).convert("RGB"))
    else:
        image = np.array(image)

    h, w, _ = image.shape

    hsv = np.array(Image.fromarray(image).convert("HSV"))
    hs = hsv[:, :, :2].reshape(-1, 2).astype(np.float32)

    kmeans = KMeans(n_clusters=n_segments, random_state=0, n_init="auto")
    labels = kmeans.fit_predict(hs)

    segment_map = labels.reshape(h, w)
    return segment_map

In [ ]:
image = np.load("../samples_of_each_environment/image_0.npy")
depth = np.load("../samples_of_each_environment/depth_0.npy")

In [7]:
seg = color_segmentation(image)

In [ ]:
def project_to_pointcloud(segment_map, depth_map, fx, fy, cx, cy):
    h, w = depth_map.shape
    u, v = np.meshgrid(np.arange(w), np.arange(h))

    z = depth_map.astype(np.float32)
    x = (u - cx) * z / fx
    y = (v - cy) * z / fy

    return np.stack([x, y, z], axis=-1).reshape(-1, 3)

In [22]:
pts = project_to_pointcloud(seg, depth[0], 4635.29022217, 4635.29022217,  960.        ,  960.        )

In [26]:
pts[:, :2]

array([[    -1.0355,     -1.0355],
       [    -1.0345,     -1.0355],
       [    -1.0334,     -1.0355],
       ...,
       [     1.0323,      1.0345],
       [     1.0334,      1.0345],
       [     1.0345,      1.0345]], shape=(3686400, 2))

In [37]:
points.shape

(7372800, 3)

In [38]:
pts = points[points[:, 2] < 4.8]

In [39]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts)
pcd = pcd.voxel_down_sample(.05)
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=10.0)
o3d.visualization.draw_plotly([pcd])

In [32]:
import numpy as np


def _unproject(depth_map, fx, fy, cx, cy):
    h, w = depth_map.shape
    u, v = np.meshgrid(np.arange(w), np.arange(h))
    z = depth_map.astype(np.float32)
    x = (u - cx) * z / fx
    y = (v - cy) * z / fy
    points = np.stack([x, y, z], axis=-1).reshape(-1, 3)
    return np.hstack([points, np.ones((len(points), 1))])


def _transform(points_h, T):
    return (T @ points_h.T).T[:, :3]


def project_to_pointcloud(depth_map1, depth_map2,
                           fx, fy, cx, cy,
                           T1=None, T2=None):
    if T1 is None: T1 = np.eye(4)
    if T2 is None: T2 = np.eye(4)

    cloud1 = _transform(_unproject(depth_map1, fx, fy, cx, cy), T1)
    cloud2 = _transform(_unproject(depth_map2, fx, fy, cx, cy), T2)

    return np.concatenate([cloud1, cloud2], axis=0)

In [35]:
points = project_to_pointcloud(depth[0], depth[1], 4635.29022217, 4635.29022217,  960.        ,  960.        , [[1., 0., 0., 0.],
 [0., 1., 0., 0.],
 [0., 0., 1., 0.],
 [0., 0., 0., 1.]],
[[1., 0., 0., 0.],
 [0., 1., 0., 0.],
 [0., 0., 1., 0.],
 [0., 0., 0., 1.]])